In [6]:
# 00. 데이터 불러오기
import pandas as pd

df_transcript_profile = pd.read_csv("transcript_profile.csv")
df_transcript_portfolio = pd.read_csv("transcript_portfolio.csv")
df_transcript_portfolio_profile = pd.read_csv("transcript_portfolio_profile.csv")

print(df_transcript_profile.shape)
print(df_transcript_portfolio.shape)
print(df_transcript_portfolio_profile.shape)

(306137, 11)
(306137, 15)
(306137, 20)


In [7]:
df_transcript_portfolio_profile.head()

,customer_id,event,time,offer_id,amount,event_reward,offer_type,offer_reward,difficulty,duration,channels,web,email,mobile,social,gender,age,income,became_member_on,profile_missing
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,0,9b98b8c7a33c4b65b9aebfe6a799e6d9,NaN,NaN,bogo,5.0,5.0,7.0,"['web', 'email', 'mobile']",1.0,1.0,1.0,0.0,F,75.0,100000.0,2017-05-09,1
1,a03223e636434f42ac4c3df47e8bac43,offer received,0,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,NaN,discount,5.0,20.0,10.0,"['web', 'email']",1.0,1.0,0.0,0.0,NaN,NaN,NaN,NaN,0
2,e2127556f4f64592b11af22de27a7932,offer received,0,2906b810c7d4411798c6938adc9daaa5,NaN,NaN,discount,2.0,10.0,7.0,"['web', 'email', 'mobile']",1.0,1.0,1.0,0.0,M,68.0,70000.0,2018-04-26,1
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,0,fafdcd668e3743c1bb461111dcafc2a4,NaN,NaN,discount,2.0,10.0,10.0,"['web', 'email', 'mobile', 'social']",1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,0
4,68617ca6246f4fbc85e91a2a49552598,offer received,0,4d5c57ea9a6940dd891ad53e9dbe8da0,NaN,NaN,bogo,10.0,10.0,5.0,"['web', 'email', 'mobile', 'social']",1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,0


In [10]:
df = df_transcript_portfolio_profile.copy()

# 정렬 (필수)
df = df.sort_values(['customer_id', 'offer_id', 'time'])

result = []

for (cust, offer), group in df.groupby(['customer_id', 'offer_id']):
    received_times = group[group['event'] == 'offer received']['time'].tolist()
    completed_times = group[group['event'] == 'offer completed']['time'].tolist()

    for c in completed_times:
        # completed 이전에 received 있는지 체크
        valid_received = [r for r in received_times if r <= c]
        
        if not valid_received:
            result.append({
                'customer_id': cust,
                'offer_id': offer,
                'completed_time': c
            })

no_received_completed = pd.DataFrame(result)

In [17]:
df['event'].value_counts()

event
transaction        138953
offer received      76277
offer viewed        57725
offer completed     33182
Name: count, dtype: int64

In [19]:
df[df['event'] == 'offer completed'].shape[0]

33182

In [20]:
print("completed 수:", len(df[df['event']=='offer completed']))
print("received 수:", len(df[df['event']=='offer received']))

completed 수: 33182
received 수: 76277


event 필터링

In [21]:
events = df[df['event'].isin(['offer received', 'offer viewed', 'offer completed'])].copy()

In [22]:
funnel = events.groupby(['customer_id', 'offer_id', 'event']).size().unstack(fill_value=0).reset_index()

In [23]:
funnel['received'] = funnel.get('offer received', 0)
funnel['viewed'] = funnel.get('offer viewed', 0)
funnel['completed'] = funnel.get('offer completed', 0)

전환고객 케이스분류 및 분석

In [24]:
# 정렬
events = events.sort_values(['customer_id', 'offer_id', 'time'])

result = []

for (cust, offer), group in events.groupby(['customer_id', 'offer_id']):
    rec_times = group[group['event'] == 'offer received']['time'].tolist()
    view_times = group[group['event'] == 'offer viewed']['time'].tolist()
    comp_times = group[group['event'] == 'offer completed']['time'].tolist()
    
    # completed 기준으로 하나씩 판단
    for comp in comp_times:
        rec_before = any(r < comp for r in rec_times)
        view_before = any(v < comp for v in view_times)
        
        if rec_before and not view_before:
            case = 'received_no_view_completed'
        elif rec_before and view_before:
            case = 'received_view_completed'
        elif (not rec_before) and view_before:
            case = 'no_received_view_completed'
        else:
            case = 'other'
        
        result.append([cust, offer, comp, case])

case_df = pd.DataFrame(result, columns=['customer_id', 'offer_id', 'time', 'case'])

In [25]:
case_df['case'].value_counts()

case
received_view_completed       22313
received_no_view_completed     9199
other                          1670
Name: count, dtype: int64

비전환고객

In [28]:
result = []

for (cust, offer), group in events.groupby(['customer_id', 'offer_id']):
    
    received_times = group[group['event'] == 'offer received']['time'].tolist()
    viewed_times = group[group['event'] == 'offer viewed']['time'].tolist()
    completed_times = group[group['event'] == 'offer completed']['time'].tolist()
    
    for r in received_times:
        
        # 다음 received 나오기 전까지만 보기 (핵심🔥)
        next_received = min([t for t in received_times if t > r], default=float('inf'))
        
        # 해당 구간 이벤트만 필터링
        viewed_in_window = [v for v in viewed_times if r < v < next_received]
        completed_in_window = [c for c in completed_times if r < c < next_received]
        
        viewed_flag = int(len(viewed_in_window) > 0)
        completed_flag = int(len(completed_in_window) > 0)
        
        # 비전환 케이스 정의
        if viewed_flag == 0 and completed_flag == 0:
            case = 'not_viewed'
        elif viewed_flag == 1 and completed_flag == 0:
            case = 'viewed_not_converted'
        else:
            case = 'other'  # 전환 or 기타
        
        result.append([cust, offer, r, viewed_flag, completed_flag, case])

In [29]:
case_df = pd.DataFrame(result, columns=[
    'customer_id', 'offer_id', 'received_time',
    'viewed', 'completed', 'case'
])

In [30]:
non_converted = case_df[case_df['case'].isin(['not_viewed', 'viewed_not_converted'])]

In [32]:
non_converted['case'].value_counts()

case
viewed_not_converted    25064
not_viewed              20208
Name: count, dtype: int64